# jC_02 - Non-IID x rasio serangan (JURNAL, Tahap 1B)

**Pertanyaan:** pada R2 (far-domain), CIC runtuh saat data makin non-IID, sedangkan UNSW datar. Naskah menduga
penyebabnya **rasio serangan**: CIC hanya ±17% serangan (kelas minoritas gampang habis di sebagian shard),
UNSW ±55% (tiap shard masih kebagian serangan). Dugaan ini belum diuji.

**Cara menguji (rasio ditukar, semua lain tetap):**

| Kondisi | Dataset | Rasio serangan (train & test) |
|---|---|---|
| `cic@orig`  | CIC  | asli (±17%) |
| `cic@55%`   | CIC  | dinaikkan ke 55% |
| `unsw@orig` | UNSW | asli (55% train / 68% test) |
| `unsw@17%`  | UNSW | diturunkan ke 17% |

Protokol sweep identik dengan R2: K=4 client, Dirichlet(alpha), FedAvg, 50 ronde, 3 seed; MCC pada test set.
Kondisi `orig` dijalankan ulang di sini supaya perbandingannya adil (pipeline sama persis).

**Kriteria dibaca (disepakati SEBELUM melihat angka)**, dengan `drop` = MCC(alpha=100) - MCC(alpha=0,1):
- Dugaan rasio serangan **didukung** bila `unsw@17%` mengalami drop >= 0,10 **dan** `cic@55%` drop <= 0,05.
- **Tidak didukung** bila pola asli bertahan (CIC tetap runtuh, UNSW tetap datar) walau rasio ditukar.
- Hasil di antaranya = sebagian; laporkan apa adanya.

Catatan: MCC bergantung pada prevalensi kelas, jadi test set ikut diubah ke rasio yang sama dengan train.

**Prasyarat:** data far-domain hasil s01 sudah ada di S3 (`unsw-far/federated_survey/data/`).
**Output:** `journal_out/r2ratio_noniid.csv` + S3 `federated_journal/ratio/results/`. Jalankan di SageMaker.

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','boto3','scipy','pandas'):
    if importlib.util.find_spec(m) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, io, json, csv, time, numpy as np, pandas as pd, boto3

BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'
SRC='unsw-far/federated_survey'      # data far-domain 9 fitur (hasil s01), sudah ada di S3
OUT='federated_journal/ratio'        # prefix hasil BARU untuk percobaan rasio serangan
OUTDIR='journal_out'; os.makedirs(OUTDIR, exist_ok=True)
s3=boto3.client('s3',region_name=REGION)

def s3_get_npz(key):
    obj=s3.get_object(Bucket=BUCKET,Key=key)
    return np.load(io.BytesIO(obj['Body'].read()),allow_pickle=True)

def load_raw():
    raw={}
    for ds in ('cic','unsw'):
        raw[ds]={}
        for sp in ('train','test'):
            d=s3_get_npz(f'{SRC}/data/{ds}/{ds}_{sp}.npz')
            raw[ds][sp]=(np.asarray(d['X'],float),np.asarray(d['y'],int))
        print(ds,'train',raw[ds]['train'][0].shape,'pos=%.3f'%raw[ds]['train'][1].mean(),
              '| test',raw[ds]['test'][0].shape,'pos=%.3f'%raw[ds]['test'][1].mean())
    return raw
RAW=load_raw()

In [ ]:
HID=[128,64]
def init_w(d, seed=42):
    rng=np.random.RandomState(seed); W=[]; L=[d]+HID+[1]
    for a,b in zip(L[:-1],L[1:]):
        W.append(rng.randn(a,b)*np.sqrt(2.0/a)); W.append(np.zeros(b))
    return W
def _relu(x): return np.maximum(0,x)
def _sig(x): return 1.0/(1.0+np.exp(-np.clip(x,-30,30)))
def fwd(W,X):
    a1=X@W[0]+W[1]; z1=_relu(a1)
    a2=z1@W[2]+W[3]; z2=_relu(a2)
    a3=z2@W[4]+W[5]
    return _sig(a3).ravel(),(X,a1,z1,a2,z2)
def train_local(W,X,y,epochs,lr,seed,batch=256):
    rng=np.random.RandomState(seed); n=len(y); W=[w.copy() for w in W]
    for _ in range(epochs):
        idx=rng.permutation(n)
        for s in range(0,n,batch):
            bi=idx[s:s+batch]; Xb=X[bi]; yb=y[bi].astype(float)
            o,(X_,a1,z1,a2,z2)=fwd(W,Xb); g=(o-yb)/max(len(yb),1)
            gW4=z2.T@g[:,None]; gb4=g.sum(0)
            d2=(g[:,None]@W[4].T)*(a2>0)
            gW2=z1.T@d2; gb2=d2.sum(0)
            d1=(d2@W[2].T)*(a1>0)
            gW0=X_.T@d1; gb0=d1.sum(0)
            for i,gv in enumerate([gW0,gb0,gW2,gb2,gW4,gb4]):
                W[i]=W[i]-lr*np.asarray(gv).reshape(W[i].shape)
    return W
def fedavg(Ws,ns):
    tot=float(sum(ns)); return [sum((ns[k]/tot)*Ws[k][i] for k in range(len(Ws))) for i in range(len(Ws[0]))]
def zfit(X):
    m=X.mean(0); s=X.std(0); s[s==0]=1.0; return m,s
def mcc(y,p):
    y=np.asarray(y).astype(int); p=np.asarray(p).astype(int)
    tp=int(((p==1)&(y==1)).sum()); tn=int(((p==0)&(y==0)).sum())
    fp=int(((p==1)&(y==0)).sum()); fn=int(((p==0)&(y==1)).sum())
    den=np.sqrt(float(tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    return (tp*tn-fp*fn)/den if den>0 else 0.0
def recall(y,p):
    y=np.asarray(y).astype(int); p=np.asarray(p).astype(int)
    tp=int(((p==1)&(y==1)).sum()); fn=int(((p==0)&(y==1)).sum())
    return tp/(tp+fn) if (tp+fn)>0 else 0.0
def dirichlet_partition(y,K,alpha,seed):
    rng=np.random.RandomState(seed); client_idx=[[] for _ in range(K)]
    for c in np.unique(y):
        idc=np.where(y==c)[0]; rng.shuffle(idc)
        props=rng.dirichlet([alpha]*K); cuts=(np.cumsum(props)*len(idc)).astype(int)[:-1]
        for k,part in enumerate(np.split(idc,cuts)): client_idx[k].extend(part.tolist())
    return [np.array(c) for c in client_idx if len(c)>0]
def resample_to_ratio(X,y,r,seed=0):
    """Ubah rasio serangan ke r dgn membuang sampel kelas yang berlebih (data tetap & deterministik)."""
    rng=np.random.RandomState(seed); pos=np.where(y==1)[0]; neg=np.where(y==0)[0]
    cur=len(pos)/len(y)
    if abs(cur-r)<1e-3: return X,y
    if cur>r:
        keep=int(round(len(neg)*r/(1-r))); pos=rng.choice(pos,size=min(keep,len(pos)),replace=False)
    else:
        keep=int(round(len(pos)*(1-r)/r)); neg=rng.choice(neg,size=min(keep,len(neg)),replace=False)
    idx=np.concatenate([pos,neg]); rng.shuffle(idx); return X[idx],y[idx]
print('util siap')

In [ ]:
CONDS={'cic@orig':('cic',None),'cic@55%':('cic',0.55),
       'unsw@orig':('unsw',None),'unsw@17%':('unsw',0.17)}
def build(cond):
    ds,r=CONDS[cond]; (Xtr,ytr),(Xte,yte)=RAW[ds]['train'],RAW[ds]['test']
    if r is not None:
        Xtr,ytr=resample_to_ratio(Xtr,ytr,r); Xte,yte=resample_to_ratio(Xte,yte,r)
    return Xtr,ytr,Xte,yte
print('%-10s %8s %8s | %8s %8s'%('kondisi','n_train','pos_tr','n_test','pos_te'))
for c in CONDS:
    Xtr,ytr,Xte,yte=build(c)
    print('%-10s %8d %8.3f | %8d %8.3f'%(c,len(ytr),ytr.mean(),len(yte),yte.mean()))

In [ ]:
ALPHAS=[100,1,0.1,0.05]; K=4; ROUNDS=50; LOCAL_EPOCHS=5; LR=3e-3; SEEDS=[13,42,101]

def run_cond(cond):
    Xtr,ytr,Xte,yte=build(cond)
    mu,sd_=zfit(Xtr); Xtr_z=(Xtr-mu)/sd_; Xte_z=(Xte-mu)/sd_
    d=Xtr.shape[1]; rows=[]
    for a in ALPHAS:
        vals=[]
        for sd in SEEDS:
            t0=time.time()
            parts=dirichlet_partition(ytr,K,a,sd); W=init_w(d,sd)
            for t in range(ROUNDS):
                Ws=[]; ns=[]
                for pidx in parts:
                    Ws.append(train_local(W,Xtr_z[pidx],ytr[pidx],LOCAL_EPOCHS,LR,sd)); ns.append(len(pidx))
                W=fedavg(Ws,ns)
            o,_=fwd(W,Xte_z); vals.append(mcc(yte,(o>=0.5).astype(int)))
            print(f'  {cond} alpha={a:<5} seed={sd:<3} MCC={vals[-1]:+.3f}  [{time.time()-t0:.0f}s]')
        rows.append(dict(cond=cond,alpha=a,mcc_mean=round(float(np.mean(vals)),4),
                         mcc_std=round(float(np.std(vals)),4),values=';'.join(str(round(v,4)) for v in vals),
                         n_train=len(ytr),ratio_train=round(float(ytr.mean()),3),ratio_test=round(float(yte.mean()),3)))
        print(f'{cond} alpha={a:<5} MCC={rows[-1]["mcc_mean"]:.3f} +/- {rows[-1]["mcc_std"]:.3f}')
    return rows

ALL=[]
for cond in CONDS:
    t0=time.time(); ALL+=run_cond(cond); print(f'== {cond} selesai [{time.time()-t0:.0f}s]\n')
path=os.path.join(OUTDIR,'r2ratio_noniid.csv')
with open(path,'w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(ALL[0].keys())); w.writeheader(); w.writerows(ALL)
s3.upload_file(path,BUCKET,f'{OUT}/results/r2ratio_noniid.csv')
print('CSV tersimpan + upload ke S3')

In [ ]:
# Ringkasan + kriteria yang disepakati sebelum melihat angka
df=pd.DataFrame(ALL)
print('=== MCC rata-rata per kondisi dan alpha ===')
print(df.pivot(index='cond',columns='alpha',values='mcc_mean').loc[list(CONDS)].round(3).to_string())
def drop(cond):
    s=df[df.cond==cond].set_index('alpha').mcc_mean
    return float(s[100]-s[0.1])
D={c:drop(c) for c in CONDS}
print('\ndrop = MCC(alpha=100) - MCC(alpha=0,1):')
for c in CONDS: print(f'  {c:10s} {D[c]:+.3f}')
print('\nKriteria (disepakati sebelum melihat angka):')
print('  unsw@17% drop >= 0.10 ?', D['unsw@17%']>=0.10)
print('  cic@55%  drop <= 0.05 ?', D['cic@55%']<=0.05)
print('Jangan menyimpulkan sebelum Kiro membaca hasilnya bersama kondisi orig.')

In [ ]:
NEED=[f'{OUT}/results/r2ratio_noniid.csv']
resp=s3.list_objects_v2(Bucket=BUCKET,Prefix=f'{OUT}/')
keys=sorted(o['Key'] for o in resp.get('Contents',[]))
print('File hasil di S3 (%d):'%len(keys))
for k in keys: print('  s3://%s/%s'%(BUCKET,k))
miss=[k for k in NEED if k not in keys]
print('LENGKAP' if not miss else 'KURANG: %s'%miss)
print()
print('Di komputer lokal (PowerShell), tarik dengan:')
print('  aws s3 sync s3://%s/%s/ ./ratio'%(BUCKET,OUT))